# Proyecto 5: Segmentación de Clientes mediante Clustering
## Notebook 3: Análisis Exploratorio de Datos (EDA)

### Objetivo
Realizar un análisis profundo de las características de los datos para:
1. Comprender los patrones de gasto por categoría
2. Identificar relaciones entre variables
3. Analizar diferencias por tipo de cliente y región
4. Determinar el número óptimo de clusters

### Metodología
- Estadísticas descriptivas multivariadas
- Visualizaciones de relaciones entre variables
- Análisis por grupos (Channel, Region)
- Evaluación de métodos para determinar k óptimo

## 1. Importación y Preparación de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Librerías importadas")

In [ ]:
# Cargar datos
ruta_datos = '../data/Wholesale customers data.csv'
df_original = pd.read_csv(ruta_datos)

# Estandarizar datos
variables_numericas = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
scaler = StandardScaler()
df_estandarizado = df_original.copy()
df_estandarizado[variables_numericas] = scaler.fit_transform(df_original[variables_numericas])

print(f"✓ Datos cargados y estandarizados")
print(f"Dimensiones: {df_estandarizado.shape}")

## 2. Análisis Univariado - Características de Gasto

In [ ]:
# Estadísticas por variable de gasto
print("\n" + "="*80)
print("ESTADÍSTICAS DE VARIABLES DE GASTO (Datos Originales)")
print("="*80 + "\n")

for var in variables_numericas:
    print(f"\n{var}:")
    print(f"  Media:        ${df_original[var].mean():>10,.0f}")
    print(f"  Mediana:      ${df_original[var].median():>10,.0f}")
    print(f"  Desv. Est.:   ${df_original[var].std():>10,.0f}")
    print(f"  Mínimo:       ${df_original[var].min():>10,.0f}")
    print(f"  Máximo:       ${df_original[var].max():>10,.0f}")
    print(f"  Rango:        ${df_original[var].max() - df_original[var].min():>10,.0f}")
    print(f"  Coef. Var.:   {(df_original[var].std()/df_original[var].mean()):>10.2f}")

## 3. Gasto Total y Proporciones por Categoría

In [ ]:
# Calcular gasto total
df_original['Gasto_Total'] = df_original[variables_numericas].sum(axis=1)

# Calcular proporciones
for var in variables_numericas:
    df_original[f'Prop_{var}'] = df_original[var] / df_original['Gasto_Total']

print("Estadísticas del Gasto Total:")
print(f"  Media:    ${df_original['Gasto_Total'].mean():,.0f}")
print(f"  Mediana:  ${df_original['Gasto_Total'].median():,.0f}")
print(f"  Rango:    ${df_original['Gasto_Total'].min():,.0f} - ${df_original['Gasto_Total'].max():,.0f}")

print("\nProporciones promedio de gasto por categoría:")
for var in variables_numericas:
    prop_media = df_original[f'Prop_{var}'].mean()
    print(f"  • {var}: {prop_media*100:>5.1f}%")

## 4. Distribuciones Detalladas de Gastos

In [ ]:
# Crear visualización detallada de distribuciones
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.ravel()

colores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']

for idx, var in enumerate(variables_numericas):
    # Histograma con distribución normal superpuesta
    axes[idx].hist(df_original[var], bins=40, color=colores[idx], alpha=0.7, edgecolor='black', density=True)
    
    # Agregar curva normal
    mu = df_original[var].mean()
    sigma = df_original[var].std()
    x = np.linspace(df_original[var].min(), df_original[var].max(), 100)
    axes[idx].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Distribución Normal')
    
    axes[idx].set_title(f'{var}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Valor ($)')
    axes[idx].set_ylabel('Densidad')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Distribuciones con curva normal superpuesta")

## 5. Análisis Bivariado - Relaciones entre Variables

In [ ]:
# Gráfico de dispersión 3D con dos categorías principales
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 6))

# Scatter plot: Fresh vs Grocery vs Milk
ax1 = fig.add_subplot(121, projection='3d')
scatter1 = ax1.scatter(df_original['Fresh'], df_original['Grocery'], df_original['Milk'], 
                        c=df_original['Gasto_Total'], cmap='viridis', s=50, alpha=0.6)
ax1.set_xlabel('Fresh ($)', fontweight='bold')
ax1.set_ylabel('Grocery ($)', fontweight='bold')
ax1.set_zlabel('Milk ($)', fontweight='bold')
ax1.set_title('Fresh vs Grocery vs Milk\n(coloreado por Gasto Total)', fontweight='bold')
plt.colorbar(scatter1, ax=ax1, label='Gasto Total')

# Scatter plot: Frozen vs Detergents_Paper
ax2 = fig.add_subplot(122)
scatter2 = ax2.scatter(df_original['Frozen'], df_original['Detergents_Paper'], 
                        c=df_original['Channel'], cmap='Set1', s=60, alpha=0.6)
ax2.set_xlabel('Frozen ($)', fontweight='bold')
ax2.set_ylabel('Detergents_Paper ($)', fontweight='bold')
ax2.set_title('Frozen vs Detergents_Paper\n(coloreado por Channel)', fontweight='bold')
cbar = plt.colorbar(scatter2, ax=ax2)
cbar.set_label('Channel (1=HoReCa, 2=Retail)')

plt.tight_layout()
plt.show()

## 6. Análisis por Channel (Tipo de Cliente)

In [ ]:
# Comparison by Channel
print("\nANÁLISIS DE GASTO POR CANAL DE VENTA\n" + "="*70)

channel_names = {1: 'Hotel/Restaurante/Café (HoReCa)', 2: 'Retail'}

for channel in sorted(df_original['Channel'].unique()):
    df_channel = df_original[df_original['Channel'] == channel]
    print(f"\n{channel_names[channel]} (n={len(df_channel)} clientes):")
    for var in variables_numericas:
        print(f"  • {var:20s}: ${df_channel[var].mean():>10,.0f} (±${df_channel[var].std():>10,.0f})")

In [ ]:
# Visualización comparativa por Channel
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, var in enumerate(variables_numericas):
    # Boxplot
    df_original.boxplot(column=var, by='Channel', ax=axes[idx])
    axes[idx].set_title(f'{var} por Canal', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Canal')
    axes[idx].set_ylabel('Gasto ($)')
    axes[idx].get_figure().suptitle('')  # Remover título automático
    
    # Etiquetar canales
    axes[idx].set_xticklabels(['HoReCa', 'Retail'])

plt.suptitle('Comparación de Gastos por Canal de Venta', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 7. Análisis por Región

In [ ]:
print("\nANÁLISIS DE GASTO POR REGIÓN\n" + "="*70)

for region in sorted(df_original['Region'].unique()):
    df_region = df_original[df_original['Region'] == region]
    print(f"\nRegión {region} (n={len(df_region)} clientes):")
    print(f"  Gasto total promedio: ${df_region['Gasto_Total'].mean():,.0f}")
    for var in variables_numericas:
        print(f"  • {var:20s}: ${df_region[var].mean():>10,.0f}")

In [ ]:
# Visualización de regiones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gasto total por región
region_gasto = df_original.groupby('Region')['Gasto_Total'].mean().sort_index()
axes[0].bar(region_gasto.index, region_gasto.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.7, edgecolor='black')
axes[0].set_title('Gasto Total Promedio por Región', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Región')
axes[0].set_ylabel('Gasto Promedio ($)')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(region_gasto.values):
    axes[0].text(i, v + 500, f'${v:,.0f}', ha='center', va='bottom', fontweight='bold')

# Cantidad de clientes por región
region_count = df_original['Region'].value_counts().sort_index()
axes[1].bar(region_count.index, region_count.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.7, edgecolor='black')
axes[1].set_title('Cantidad de Clientes por Región', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Región')
axes[1].set_ylabel('Cantidad')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(region_count.values):
    axes[1].text(i, v + 5, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Matriz de Correlación Detallada

In [ ]:
# Calcular correlación
correlacion = df_estandarizado[variables_numericas].corr()

print("\nMATRIZ DE CORRELACIÓN:")
print(correlacion.round(3))

# Identificar pares más correlacionados
print("\nPares de variables MÁS correlacionados:")
corr_alto = []
for i in range(len(variables_numericas)):
    for j in range(i+1, len(variables_numericas)):
        corr_val = correlacion.iloc[i, j]
        corr_alto.append((variables_numericas[i], variables_numericas[j], corr_val))

corr_alto.sort(key=lambda x: abs(x[2]), reverse=True)
for var1, var2, corr_val in corr_alto[:5]:
    print(f"  • {var1} - {var2}: {corr_val:>6.3f}")

In [ ]:
# Visualizar matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, cbar_kws={"shrink": 0.8}, 
            fmt='.2f', vmin=-1, vmax=1)
plt.title('Matriz de Correlación - Variables de Gasto\n(Datos Estandarizados)', 
         fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 9. PCA (Análisis de Componentes Principales) para Visualización

In [ ]:
# Aplicar PCA
pca = PCA()
pca_data = pca.fit_transform(df_estandarizado[variables_numericas])

# Varianza explicada
varianza_acumulada = np.cumsum(pca.explained_variance_ratio_)

print("\nANÁLISIS PCA:")
print(f"Varianza explicada por componente:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var*100:>5.2f}% | Acumulada: {varianza_acumulada[i]*100:>5.2f}%")

print(f"\nPara explicar 80% de varianza se necesitan {np.argmax(varianza_acumulada >= 0.80) + 1} componentes")
print(f"Para explicar 90% de varianza se necesitan {np.argmax(varianza_acumulada >= 0.90) + 1} componentes")

In [ ]:
# Visualizar varianza explicada por PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), 
            pca.explained_variance_ratio_ * 100, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Componente Principal', fontweight='bold')
axes[0].set_ylabel('Varianza Explicada (%)', fontweight='bold')
axes[0].set_title('Scree Plot - Varianza por Componente', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Varianza acumulada
axes[1].plot(range(1, len(varianza_acumulada) + 1), varianza_acumulada * 100, 
            'o-', linewidth=2, markersize=8, color='steelblue')
axes[1].axhline(y=80, color='red', linestyle='--', linewidth=2, label='80% Varianza')
axes[1].axhline(y=90, color='orange', linestyle='--', linewidth=2, label='90% Varianza')
axes[1].set_xlabel('Número de Componentes', fontweight='bold')
axes[1].set_ylabel('Varianza Acumulada (%)', fontweight='bold')
axes[1].set_title('Varianza Acumulada - PCA', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualizar en 2D usando los primeros 2 componentes
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC1 vs PC2 coloreado por Channel
scatter1 = axes[0].scatter(pca_data[:, 0], pca_data[:, 1], 
                           c=df_original['Channel'], cmap='Set1', s=60, alpha=0.6, edgecolors='black')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
axes[0].set_title('PCA: PC1 vs PC2 (coloreado por Channel)', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('Channel (1=HoReCa, 2=Retail)')

# PC1 vs PC2 coloreado por Región
scatter2 = axes[1].scatter(pca_data[:, 0], pca_data[:, 1], 
                           c=df_original['Region'], cmap='viridis', s=60, alpha=0.6, edgecolors='black')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)', fontweight='bold')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)', fontweight='bold')
axes[1].set_title('PCA: PC1 vs PC2 (coloreado por Región)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)
cbar2 = plt.colorbar(scatter2, ax=axes[1])
cbar2.set_label('Región')

plt.tight_layout()
plt.show()

## 10. Determinación del Número Óptimo de Clusters

### Métodos para determinar k:
1. **Elbow Method**: Gráfico de inercia vs número de clusters
2. **Silhouette Score**: Mide qué tan bien separados están los clusters
3. **Davies-Bouldin Index**: Relación entre dispersión intra-cluster e inter-cluster

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Calcular métricas para diferentes valores de k
k_values = range(2, 11)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []

print("\nEVALUACIÓN DE NÚMERO DE CLUSTERS:\n" + "="*70)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(df_estandarizado[variables_numericas])
    
    inertia = kmeans.inertia_
    silhouette = silhouette_score(df_estandarizado[variables_numericas], labels)
    davies_bouldin = davies_bouldin_score(df_estandarizado[variables_numericas], labels)
    
    inertias.append(inertia)
    silhouette_scores.append(silhouette)
    davies_bouldin_scores.append(davies_bouldin)
    
    print(f"k={k} | Inercia: {inertia:>10.0f} | Silhouette: {silhouette:>6.3f} | Davies-Bouldin: {davies_bouldin:>6.3f}")

In [ ]:
# Visualizar métodos para determinar k óptimo
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Elbow Method
axes[0].plot(k_values, inertias, 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].set_xlabel('Número de Clusters (k)', fontweight='bold')
axes[0].set_ylabel('Inercia', fontweight='bold')
axes[0].set_title('Elbow Method', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_xticks(k_values)

# Silhouette Score (mayor es mejor)
best_k_silhouette = k_values[np.argmax(silhouette_scores)]
axes[1].plot(k_values, silhouette_scores, 'o-', linewidth=2, markersize=8, color='green')
axes[1].axvline(x=best_k_silhouette, color='red', linestyle='--', linewidth=2, label=f'k={best_k_silhouette}')
axes[1].set_xlabel('Número de Clusters (k)', fontweight='bold')
axes[1].set_ylabel('Silhouette Score', fontweight='bold')
axes[1].set_title('Silhouette Score (↑ mejor)', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].legend()
axes[1].set_xticks(k_values)

# Davies-Bouldin Index (menor es mejor)
best_k_db = k_values[np.argmin(davies_bouldin_scores)]
axes[2].plot(k_values, davies_bouldin_scores, 'o-', linewidth=2, markersize=8, color='orange')
axes[2].axvline(x=best_k_db, color='red', linestyle='--', linewidth=2, label=f'k={best_k_db}')
axes[2].set_xlabel('Número de Clusters (k)', fontweight='bold')
axes[2].set_ylabel('Davies-Bouldin Index', fontweight='bold')
axes[2].set_title('Davies-Bouldin Index (↓ mejor)', fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)
axes[2].legend()
axes[2].set_xticks(k_values)

plt.tight_layout()
plt.show()

print(f"\n✓ Recomendaciones:")
print(f"  • Según Silhouette Score: k={best_k_silhouette}")
print(f"  • Según Davies-Bouldin: k={best_k_db}")
print(f"  • Consideración visual (Elbow): k=3-4")

## 11. Resumen del EDA

In [ ]:
resumen_eda = f"""
{'='*80}
RESUMEN - ANÁLISIS EXPLORATORIO DE DATOS
{'='*80}

1. CARACTERÍSTICAS GENERALES:
   • {df_original.shape[0]} clientes mayoristas analizados
   • 6 categorías de gasto: Fresh, Milk, Grocery, Frozen, Detergents_Paper, Delicassen
   • 2 tipos de cliente: HoReCa (n=142) y Retail (n={len(df_original[df_original['Channel']==2])})
   • 3 regiones geográficas

2. DISTRIBUCIONES DE GASTOS:
   • Grocery: Mayor gasto promedio (${df_original['Grocery'].mean():,.0f})
   • Fresh: Segundo mayor gasto (${df_original['Fresh'].mean():,.0f})
   • Frozen: Menor gasto promedio (${df_original['Frozen'].mean():,.0f})
   • Todas las variables tienen distribuciones asimétricas (positivas)

3. DIFERENCIAS POR CANAL:
   • HoReCa: Mayor gasto en Fresh (productos perecederos)
   • Retail: Mayor gasto en Grocery (comestibles de larga duración)
   • Patrón diferenciado sugiere segmentación natural por canal

4. CORRELACIONES:
   • Grocery-Detergents_Paper: Fuerte correlación ({correlacion.loc['Grocery', 'Detergents_Paper']:.3f})
   • Fresh-Frozen: Correlación moderada
   • Variables tienen alta multicolinealidad

5. ANÁLISIS PCA:
   • 2 primeros componentes explican {varianza_acumulada[1]*100:.1f}% de varianza
   • 3 primeros componentes explican {varianza_acumulada[2]*100:.1f}% de varianza
   • Datos altamente multidimensionales

6. NÚMERO ÓPTIMO DE CLUSTERS (k):
   • Silhouette Score sugiere: k={best_k_silhouette}
   • Davies-Bouldin sugiere: k={best_k_db}
   • Visualmente (elbow): k=3-4 es razonable
   • Recomendación: Probar k=3, 4, 5 en los siguientes notebooks

7. CONCLUSIONES:
   ✓ Datos listos para clustering
   ✓ Existen patrones diferenciados en los datos
   ✓ Se espera encontrar 3-4 segmentos claros de clientes
   ✓ Diferenciación por tipo de cliente (HoReCa vs Retail) es clara

{'='*80}
"""

print(resumen_eda)

## 12. Próximas Etapas

En los siguientes notebooks implementaremos:

**Notebook 4:** K-means Clustering
- Aplicación del algoritmo K-means
- Evaluación de diferentes valores de k
- Análisis de centros de clusters

**Notebook 5:** Clustering Jerárquico
- Algoritmos aglomerativos
- Dendrogramas
- Comparación con K-means

**Notebook 6:** DBSCAN
- Clustering basado en densidad
- Detección de outliers
- Ventajas y limitaciones

**Notebook 7:** Validación e Interpretación
- Métricas de calidad de clusters
- Perfiles característicos
- Recomendaciones de marketing

In [ ]:
print("✓ Análisis Exploratorio completado exitosamente")
print("✓ Los datos están listos para aplicación de algoritmos de clustering")